# Gold Hiring Trends Mart

**Purpose**: Dashboard-ready hiring trends with temporal aggregations and rollup dimensions.

**Target Table**: `workspace.gold.gold_hiring_trends`

**Grain**: Multiple aggregation levels - daily by sector, role, location, and company

**Aggregation Levels**:
- `sector`: Daily aggregated by sector (all roles, all locations)
- `role`: Daily aggregated by role (all sectors, all locations)
- `location`: Daily aggregated by location (all sectors, all roles)
- `company`: Daily aggregated by company within sector
- `daily`: Full granularity (all dimensions)

**Key Metrics**: Active jobs, new jobs, closed jobs, net change, temporal comparisons

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_hiring_trends
USING DELTA
COMMENT 'Hiring trends mart with multiple aggregation levels for dashboards'
AS

WITH base_metrics AS (
  SELECT
    f.posting_date_sk,
    f.sector_sk,
    f.role_sk,
    f.location_sk,
    f.company_sk,
    
    -- MEASURES
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS active_jobs_count,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) AS new_jobs_count,
    COUNT(CASE WHEN f.is_update = TRUE THEN 1 END) AS updated_jobs_count,
    COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS closed_jobs_count,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) - 
      COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS net_job_change,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.company_sk END) AS unique_companies_hiring,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.role_sk END) AS unique_roles_available
    
  FROM workspace.warehouse.fact_job_postings f
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
  GROUP BY f.posting_date_sk, f.sector_sk, f.role_sk, f.location_sk, f.company_sk
),

-- Aggregation Level: Daily by Sector
sector_agg AS (
  SELECT
    posting_date_sk AS hiring_date_sk,
    sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    CAST(NULL AS BIGINT) AS location_sk,
    CAST(NULL AS BIGINT) AS company_sk,
    'sector' AS aggregation_level,
    SUM(active_jobs_count) AS active_jobs_count,
    SUM(new_jobs_count) AS new_jobs_count,
    SUM(updated_jobs_count) AS updated_jobs_count,
    SUM(closed_jobs_count) AS closed_jobs_count,
    SUM(net_job_change) AS net_job_change,
    MAX(unique_companies_hiring) AS unique_companies_hiring,
    MAX(unique_roles_available) AS unique_roles_available
  FROM base_metrics
  GROUP BY posting_date_sk, sector_sk
),

-- Aggregation Level: Daily by Role
role_agg AS (
  SELECT
    posting_date_sk AS hiring_date_sk,
    CAST(-1 AS BIGINT) AS sector_sk,
    role_sk,
    CAST(NULL AS BIGINT) AS location_sk,
    CAST(NULL AS BIGINT) AS company_sk,
    'role' AS aggregation_level,
    SUM(active_jobs_count) AS active_jobs_count,
    SUM(new_jobs_count) AS new_jobs_count,
    SUM(updated_jobs_count) AS updated_jobs_count,
    SUM(closed_jobs_count) AS closed_jobs_count,
    SUM(net_job_change) AS net_job_change,
    MAX(unique_companies_hiring) AS unique_companies_hiring,
    COUNT(DISTINCT sector_sk) AS unique_roles_available
  FROM base_metrics
  GROUP BY posting_date_sk, role_sk
),

-- Aggregation Level: Daily by Location
location_agg AS (
  SELECT
    posting_date_sk AS hiring_date_sk,
    CAST(-1 AS BIGINT) AS sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    location_sk,
    CAST(NULL AS BIGINT) AS company_sk,
    'location' AS aggregation_level,
    SUM(active_jobs_count) AS active_jobs_count,
    SUM(new_jobs_count) AS new_jobs_count,
    SUM(updated_jobs_count) AS updated_jobs_count,
    SUM(closed_jobs_count) AS closed_jobs_count,
    SUM(net_job_change) AS net_job_change,
    MAX(unique_companies_hiring) AS unique_companies_hiring,
    MAX(unique_roles_available) AS unique_roles_available
  FROM base_metrics
  GROUP BY posting_date_sk, location_sk
),

-- Aggregation Level: Daily by Company
company_agg AS (
  SELECT
    posting_date_sk AS hiring_date_sk,
    sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    CAST(NULL AS BIGINT) AS location_sk,
    company_sk,
    'company' AS aggregation_level,
    SUM(active_jobs_count) AS active_jobs_count,
    SUM(new_jobs_count) AS new_jobs_count,
    SUM(updated_jobs_count) AS updated_jobs_count,
    SUM(closed_jobs_count) AS closed_jobs_count,
    SUM(net_job_change) AS net_job_change,
    1 AS unique_companies_hiring,
    MAX(unique_roles_available) AS unique_roles_available
  FROM base_metrics
  GROUP BY posting_date_sk, sector_sk, company_sk
),

-- Combine all aggregation levels
combined_agg AS (
  SELECT * FROM sector_agg
  UNION ALL
  SELECT * FROM role_agg
  UNION ALL
  SELECT * FROM location_agg
  UNION ALL
  SELECT * FROM company_agg
),

-- Add temporal metrics
temporal_enriched AS (
  SELECT
    ca.*,
    
    -- Prior period comparisons
    LAG(ca.active_jobs_count, 1) OVER (
      PARTITION BY ca.aggregation_level, ca.sector_sk, ca.role_sk, ca.location_sk, ca.company_sk
      ORDER BY ca.hiring_date_sk
    ) AS prev_period_active_jobs,
    
    -- Week-to-date cumulative
    SUM(ca.new_jobs_count) OVER (
      PARTITION BY ca.aggregation_level, ca.sector_sk, ca.role_sk, ca.location_sk, ca.company_sk,
        YEAR(TO_DATE(CAST(ca.hiring_date_sk AS STRING), 'yyyyMMdd')),
        WEEKOFYEAR(TO_DATE(CAST(ca.hiring_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY ca.hiring_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS wtd_new_jobs,
    
    -- Month-to-date cumulative
    SUM(ca.new_jobs_count) OVER (
      PARTITION BY ca.aggregation_level, ca.sector_sk, ca.role_sk, ca.location_sk, ca.company_sk,
        YEAR(TO_DATE(CAST(ca.hiring_date_sk AS STRING), 'yyyyMMdd')),
        MONTH(TO_DATE(CAST(ca.hiring_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY ca.hiring_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS mtd_new_jobs
    
  FROM combined_agg ca
)

SELECT
  -- DIMENSIONS
  te.hiring_date_sk,
  te.sector_sk,
  te.role_sk,
  te.location_sk,
  te.company_sk,
  te.aggregation_level,
  
  -- MEASURES
  te.active_jobs_count,
  te.new_jobs_count,
  te.updated_jobs_count,
  te.closed_jobs_count,
  te.net_job_change,
  te.unique_companies_hiring,
  te.unique_roles_available,
  
  -- TEMPORAL METRICS
  te.wtd_new_jobs,
  te.mtd_new_jobs,
  te.prev_period_active_jobs,
  
  -- DERIVED: Percent change
  CASE 
    WHEN te.prev_period_active_jobs > 0 
    THEN CAST((te.active_jobs_count - te.prev_period_active_jobs) AS DECIMAL(10,4)) / te.prev_period_active_jobs
    ELSE NULL 
  END AS pct_change_vs_prev_period,
  
  -- DERIVED: Job churn rate
  CASE 
    WHEN te.prev_period_active_jobs > 0 
    THEN CAST(te.closed_jobs_count AS DECIMAL(10,4)) / te.prev_period_active_jobs
    ELSE NULL 
  END AS job_churn_rate,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS created_at,
  UUID() AS batch_id
  
FROM temporal_enriched te
ORDER BY te.aggregation_level, te.hiring_date_sk DESC;

In [0]:
%sql
-- Validation: Row counts by aggregation level
SELECT
  aggregation_level,
  COUNT(*) AS row_count,
  MIN(hiring_date_sk) AS min_date,
  MAX(hiring_date_sk) AS max_date,
  ROUND(AVG(active_jobs_count), 2) AS avg_active_jobs,
  ROUND(AVG(new_jobs_count), 2) AS avg_new_jobs
FROM workspace.gold.gold_hiring_trends
GROUP BY aggregation_level
ORDER BY aggregation_level;

# Gold Hiring Trends Mart

**Purpose**: Dashboard-ready hiring trends mart with explicit measures, dimensions, and temporal metrics for Superset visualization.

**Target Table**: `workspace.gold.gold_hiring_trends`

## Schema Structure

### Dimensions
- `hiring_date_sk` BIGINT - Date key in YYYYMMDD format
- `sector_sk` BIGINT - FK to dim_sector (-1 for all sectors rollup)
- `role_sk` BIGINT - FK to dim_role (NULL for all roles rollup)
- `location_sk` BIGINT - FK to dim_location (NULL for all locations rollup)
- `company_sk` BIGINT - FK to dim_company (NULL for all companies rollup)
- `aggregation_level` STRING - Aggregation grain ('daily', 'sector', 'role', 'location', 'company')

### Measures
- `active_jobs_count` BIGINT - Count of active job postings
- `new_jobs_count` BIGINT - Count of new jobs posted
- `updated_jobs_count` BIGINT - Count of updated jobs
- `closed_jobs_count` BIGINT - Jobs that became inactive
- `net_job_change` BIGINT - New jobs minus closed jobs
- `unique_companies_hiring` BIGINT - Distinct companies with active jobs
- `unique_roles_available` BIGINT - Distinct roles posted
- `job_churn_rate` DECIMAL(10,4) - Closed jobs / prior period active jobs

### Temporal Metrics
- `wtd_new_jobs` BIGINT - Week-to-date new jobs
- `mtd_new_jobs` BIGINT - Month-to-date new jobs
- `prev_period_active_jobs` BIGINT - Active jobs from prior day
- `pct_change_vs_prev_period` DECIMAL(10,4) - Percent change vs prior period

### Metadata
- `created_at` TIMESTAMP
- `batch_id` STRING

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_hiring_trends (
  -- Dimensions
  hiring_date_sk BIGINT COMMENT 'Date key YYYYMMDD',
  sector_sk BIGINT COMMENT 'FK to dim_sector (-1 for all sectors rollup)',
  role_sk BIGINT COMMENT 'FK to dim_role (NULL for all roles rollup)',
  location_sk BIGINT COMMENT 'FK to dim_location (NULL for all locations rollup)',
  company_sk BIGINT COMMENT 'FK to dim_company (NULL for all companies rollup)',
  aggregation_level STRING COMMENT 'Aggregation grain',
  
  -- Measures
  active_jobs_count BIGINT COMMENT 'Count of active job postings',
  new_jobs_count BIGINT COMMENT 'Count of new jobs posted',
  updated_jobs_count BIGINT COMMENT 'Count of updated jobs',
  closed_jobs_count BIGINT COMMENT 'Jobs that became inactive',
  net_job_change BIGINT COMMENT 'New jobs minus closed jobs',
  unique_companies_hiring BIGINT COMMENT 'Distinct companies with active jobs',
  unique_roles_available BIGINT COMMENT 'Distinct roles posted',
  job_churn_rate DECIMAL(10,4) COMMENT 'Closed jobs / prior period active jobs',
  
  -- Temporal Metrics
  wtd_new_jobs BIGINT COMMENT 'Week-to-date new jobs',
  mtd_new_jobs BIGINT COMMENT 'Month-to-date new jobs',
  prev_period_active_jobs BIGINT COMMENT 'Active jobs from prior day',
  pct_change_vs_prev_period DECIMAL(10,4) COMMENT 'Percent change vs prior period',
  
  -- Metadata
  created_at TIMESTAMP COMMENT 'Record creation timestamp',
  batch_id STRING COMMENT 'Batch identifier'
)
USING DELTA
COMMENT 'Gold layer hiring trends mart for dashboard consumption';

In [0]:
%sql
-- Base aggregation CTE: Daily metrics by all dimensions
WITH base_metrics AS (
  SELECT
    posting_date_sk AS hiring_date_sk,
    sector_sk,
    role_sk,
    location_sk,
    company_sk,
    'daily' AS aggregation_level,
    
    -- Core measures
    COUNT(CASE WHEN active_flag = TRUE THEN 1 END) AS active_jobs_count,
    COUNT(CASE WHEN is_new_job = TRUE THEN 1 END) AS new_jobs_count,
    COUNT(CASE WHEN is_update = TRUE THEN 1 END) AS updated_jobs_count,
    COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END) AS closed_jobs_count,
    COUNT(CASE WHEN is_new_job = TRUE THEN 1 END) - COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END) AS net_job_change,
    COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN company_sk END) AS unique_companies_hiring,
    COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN role_sk END) AS unique_roles_available
    
  FROM workspace.warehouse.fact_job_postings
  WHERE posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
  GROUP BY
    posting_date_sk,
    sector_sk,
    role_sk,
    location_sk,
    company_sk
),

-- Add temporal metrics using window functions
temporal_metrics AS (
  SELECT
    *,
    -- Prior period comparison
    LAG(active_jobs_count, 1) OVER (
      PARTITION BY sector_sk, role_sk, location_sk, company_sk 
      ORDER BY hiring_date_sk
    ) AS prev_period_active_jobs,
    
    -- Week-to-date cumulative
    SUM(new_jobs_count) OVER (
      PARTITION BY sector_sk, role_sk, location_sk, company_sk, 
                   YEAR(TO_DATE(CAST(hiring_date_sk AS STRING), 'yyyyMMdd')),
                   WEEKOFYEAR(TO_DATE(CAST(hiring_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY hiring_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS wtd_new_jobs,
    
    -- Month-to-date cumulative
    SUM(new_jobs_count) OVER (
      PARTITION BY sector_sk, role_sk, location_sk, company_sk,
                   YEAR(TO_DATE(CAST(hiring_date_sk AS STRING), 'yyyyMMdd')),
                   MONTH(TO_DATE(CAST(hiring_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY hiring_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS mtd_new_jobs
    
  FROM base_metrics
),

-- Calculate derived metrics
final_daily AS (
  SELECT
    hiring_date_sk,
    sector_sk,
    role_sk,
    location_sk,
    company_sk,
    aggregation_level,
    active_jobs_count,
    new_jobs_count,
    updated_jobs_count,
    closed_jobs_count,
    net_job_change,
    unique_companies_hiring,
    unique_roles_available,
    CAST(closed_jobs_count AS DECIMAL(10,4)) / NULLIF(prev_period_active_jobs, 0) AS job_churn_rate,
    wtd_new_jobs,
    mtd_new_jobs,
    prev_period_active_jobs,
    CAST((active_jobs_count - prev_period_active_jobs) AS DECIMAL(10,4)) / NULLIF(prev_period_active_jobs, 0) AS pct_change_vs_prev_period,
    CURRENT_TIMESTAMP() AS created_at,
    CAST(uuid() AS STRING) AS batch_id
  FROM temporal_metrics
)

SELECT * FROM final_daily;

In [0]:
%sql
-- Sector rollup: Daily by sector (all roles, all locations, all companies)
WITH sector_rollup AS (
  SELECT
    posting_date_sk AS hiring_date_sk,
    sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    CAST(NULL AS BIGINT) AS location_sk,
    CAST(NULL AS BIGINT) AS company_sk,
    'sector' AS aggregation_level,
    
    COUNT(CASE WHEN active_flag = TRUE THEN 1 END) AS active_jobs_count,
    COUNT(CASE WHEN is_new_job = TRUE THEN 1 END) AS new_jobs_count,
    COUNT(CASE WHEN is_update = TRUE THEN 1 END) AS updated_jobs_count,
    COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END) AS closed_jobs_count,
    COUNT(CASE WHEN is_new_job = TRUE THEN 1 END) - COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END) AS net_job_change,
    COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN company_sk END) AS unique_companies_hiring,
    COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN role_sk END) AS unique_roles_available
    
  FROM workspace.warehouse.fact_job_postings
  WHERE posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
  GROUP BY posting_date_sk, sector_sk
),

temporal_metrics AS (
  SELECT
    *,
    LAG(active_jobs_count, 1) OVER (PARTITION BY sector_sk ORDER BY hiring_date_sk) AS prev_period_active_jobs,
    SUM(new_jobs_count) OVER (
      PARTITION BY sector_sk, 
                   YEAR(TO_DATE(CAST(hiring_date_sk AS STRING), 'yyyyMMdd')),
                   WEEKOFYEAR(TO_DATE(CAST(hiring_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY hiring_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS wtd_new_jobs,
    SUM(new_jobs_count) OVER (
      PARTITION BY sector_sk,
                   YEAR(TO_DATE(CAST(hiring_date_sk AS STRING), 'yyyyMMdd')),
                   MONTH(TO_DATE(CAST(hiring_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY hiring_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS mtd_new_jobs
  FROM sector_rollup
)

SELECT
  hiring_date_sk,
  sector_sk,
  role_sk,
  location_sk,
  company_sk,
  aggregation_level,
  active_jobs_count,
  new_jobs_count,
  updated_jobs_count,
  closed_jobs_count,
  net_job_change,
  unique_companies_hiring,
  unique_roles_available,
  CAST(closed_jobs_count AS DECIMAL(10,4)) / NULLIF(prev_period_active_jobs, 0) AS job_churn_rate,
  wtd_new_jobs,
  mtd_new_jobs,
  prev_period_active_jobs,
  CAST((active_jobs_count - prev_period_active_jobs) AS DECIMAL(10,4)) / NULLIF(prev_period_active_jobs, 0) AS pct_change_vs_prev_period,
  CURRENT_TIMESTAMP() AS created_at,
  CAST(uuid() AS STRING) AS batch_id
FROM temporal_metrics;

In [0]:
%sql
-- Complete INSERT combining all aggregation levels
INSERT INTO workspace.gold.gold_hiring_trends

-- Daily grain (all dimensions)
SELECT
  posting_date_sk AS hiring_date_sk,
  sector_sk, role_sk, location_sk, company_sk,
  'daily' AS aggregation_level,
  COUNT(CASE WHEN active_flag = TRUE THEN 1 END) AS active_jobs_count,
  COUNT(CASE WHEN is_new_job = TRUE THEN 1 END) AS new_jobs_count,
  COUNT(CASE WHEN is_update = TRUE THEN 1 END) AS updated_jobs_count,
  COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END) AS closed_jobs_count,
  COUNT(CASE WHEN is_new_job = TRUE THEN 1 END) - COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END) AS net_job_change,
  COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN company_sk END) AS unique_companies_hiring,
  COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN role_sk END) AS unique_roles_available,
  CAST(0 AS DECIMAL(10,4)) AS job_churn_rate, -- Will update with window function
  CAST(0 AS BIGINT) AS wtd_new_jobs, -- Will update
  CAST(0 AS BIGINT) AS mtd_new_jobs, -- Will update
  CAST(0 AS BIGINT) AS prev_period_active_jobs, -- Will update
  CAST(0 AS DECIMAL(10,4)) AS pct_change_vs_prev_period, -- Will update
  CURRENT_TIMESTAMP() AS created_at,
  CAST(uuid() AS STRING) AS batch_id
FROM workspace.warehouse.fact_job_postings
WHERE posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
GROUP BY posting_date_sk, sector_sk, role_sk, location_sk, company_sk

UNION ALL

-- Sector rollup
SELECT
  posting_date_sk, sector_sk, NULL, NULL, NULL, 'sector',
  COUNT(CASE WHEN active_flag = TRUE THEN 1 END),
  COUNT(CASE WHEN is_new_job = TRUE THEN 1 END),
  COUNT(CASE WHEN is_update = TRUE THEN 1 END),
  COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END),
  COUNT(CASE WHEN is_new_job = TRUE THEN 1 END) - COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END),
  COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN company_sk END),
  COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN role_sk END),
  0, 0, 0, 0, 0,
  CURRENT_TIMESTAMP(), CAST(uuid() AS STRING)
FROM workspace.warehouse.fact_job_postings
WHERE posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
GROUP BY posting_date_sk, sector_sk

UNION ALL

-- Role rollup
SELECT
  posting_date_sk, -1, role_sk, NULL, NULL, 'role',
  COUNT(CASE WHEN active_flag = TRUE THEN 1 END),
  COUNT(CASE WHEN is_new_job = TRUE THEN 1 END),
  COUNT(CASE WHEN is_update = TRUE THEN 1 END),
  COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END),
  COUNT(CASE WHEN is_new_job = TRUE THEN 1 END) - COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END),
  COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN company_sk END),
  COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN role_sk END),
  0, 0, 0, 0, 0,
  CURRENT_TIMESTAMP(), CAST(uuid() AS STRING)
FROM workspace.warehouse.fact_job_postings
WHERE posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
GROUP BY posting_date_sk, role_sk

UNION ALL

-- Location rollup
SELECT
  posting_date_sk, -1, NULL, location_sk, NULL, 'location',
  COUNT(CASE WHEN active_flag = TRUE THEN 1 END),
  COUNT(CASE WHEN is_new_job = TRUE THEN 1 END),
  COUNT(CASE WHEN is_update = TRUE THEN 1 END),
  COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END),
  COUNT(CASE WHEN is_new_job = TRUE THEN 1 END) - COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END),
  COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN company_sk END),
  COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN role_sk END),
  0, 0, 0, 0, 0,
  CURRENT_TIMESTAMP(), CAST(uuid() AS STRING)
FROM workspace.warehouse.fact_job_postings
WHERE posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
GROUP BY posting_date_sk, location_sk

UNION ALL

-- Company rollup
SELECT
  posting_date_sk, sector_sk, NULL, NULL, company_sk, 'company',
  COUNT(CASE WHEN active_flag = TRUE THEN 1 END),
  COUNT(CASE WHEN is_new_job = TRUE THEN 1 END),
  COUNT(CASE WHEN is_update = TRUE THEN 1 END),
  COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END),
  COUNT(CASE WHEN is_new_job = TRUE THEN 1 END) - COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END),
  COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN company_sk END),
  COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN role_sk END),
  0, 0, 0, 0, 0,
  CURRENT_TIMESTAMP(), CAST(uuid() AS STRING)
FROM workspace.warehouse.fact_job_postings
WHERE posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
GROUP BY posting_date_sk, sector_sk, company_sk;

In [0]:
%sql
-- Validate the gold_hiring_trends table
SELECT
  aggregation_level,
  COUNT(*) AS row_count,
  MIN(hiring_date_sk) AS min_date,
  MAX(hiring_date_sk) AS max_date,
  SUM(active_jobs_count) AS total_active_jobs,
  SUM(new_jobs_count) AS total_new_jobs,
  SUM(closed_jobs_count) AS total_closed_jobs
FROM workspace.gold.gold_hiring_trends
GROUP BY aggregation_level
ORDER BY aggregation_level;